<a href="https://colab.research.google.com/github/ishani6/egfr-drug-discovery-app/blob/main/ChEMBL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install chembl_webresource_client
!pip -q install rdkit
!pip -q install pandas
!pip -q install requests
!pip install torch_geometric

In [ ]:
import pandas as pd
from chembl_webresource_client.new_client import new_client

TARGET_ID = "CHEMBL203"
STEP = 1000

target = new_client.target
activity = new_client.activity

target_data = target.get(TARGET_ID)
print(target_data.get("pref_name"), target_data.get("organism"), target_data.get("target_type"))

query = activity.filter(
    target_chembl_id=TARGET_ID,
    standard_type="IC50"
).only([
    "activity_id",
    "molecule_chembl_id",
    "standard_type",
    "standard_value",
    "standard_units",
    "assay_chembl_id",
    "document_chembl_id"
])

rows = []
for start in range(0, 1000000, STEP):
    chunk = list(query[start:start + STEP])
    if not chunk:
        break
    rows.extend(chunk)
    print("Loaded:", len(rows))

acts = pd.DataFrame(rows)
print("Final rows:", len(acts))
acts.head()

In [ ]:
OUTFILE = "egfr_ic50_full.csv"
acts.to_csv(OUTFILE, index=False)
print("Saved:", OUTFILE)
print("Rows:", len(acts))
print(acts.head())

In [ ]:
import requests

INPUT_FILE = "egfr_ic50_full.csv"
OUTPUT_FILE = "egfr_nsclc_only.csv"
PATTERN = r"nsclc|non-small cell lung cancer|non-small-cell lung cancer|lung adenocarcinoma|egfr-mutant lung cancer"

df = pd.read_csv(INPUT_FILE)

session = requests.Session()

def fetch_batch(resource, ids, id_field, fields):
    if not ids:
        return pd.DataFrame()
    url = f"https://www.ebi.ac.uk/chembl/api/data/{resource}.json"
    params = {f"{id_field}__in": ",".join(map(str, ids)), "limit": len(ids)}
    r = session.get(url, params=params, timeout=60)
    r.raise_for_status()
    data = r.json()
    key = list(data.keys())[0]
    rows = data[key]
    return pd.DataFrame([{k: row.get(k) for k in fields} for row in rows])

# Batch fetch assay metadata
assay_ids = df["assay_chembl_id"].dropna().astype(str).unique().tolist()
assay_df = pd.concat([
    fetch_batch("assay", assay_ids[i:i+100], "assay_chembl_id",
                ["assay_chembl_id", "description", "assay_type", "assay_organism"])
    for i in range(0, len(assay_ids), 100)
], ignore_index=True) if assay_ids else pd.DataFrame()

if not assay_df.empty:
    assay_df = assay_df.rename(columns={"description": "assay_description"})

# Batch fetch document metadata
doc_ids = df["document_chembl_id"].dropna().astype(str).unique().tolist()
doc_df = pd.concat([
    fetch_batch("document", doc_ids[i:i+100], "document_chembl_id",
                ["document_chembl_id", "title", "abstract"])
    for i in range(0, len(doc_ids), 100)
], ignore_index=True) if doc_ids else pd.DataFrame()

if not doc_df.empty:
    doc_df = doc_df.rename(columns={"title": "doc_title", "abstract": "doc_abstract"})

# Merge
if not assay_df.empty:
    df = df.merge(assay_df, on="assay_chembl_id", how="left")
if not doc_df.empty:
    df = df.merge(doc_df, on="document_chembl_id", how="left")

# NSCLC filter
mask = (
    df.get("assay_description", pd.Series("", index=df.index)).fillna("").str.contains(PATTERN, case=False, regex=True) |
    df.get("doc_title", pd.Series("", index=df.index)).fillna("").str.contains(PATTERN, case=False, regex=True) |
    df.get("doc_abstract", pd.Series("", index=df.index)).fillna("").str.contains(PATTERN, case=False, regex=True)
)

df = df[mask].copy()
df["standard_value"] = pd.to_numeric(df["standard_value"], errors="coerce")
df = df.dropna(subset=["standard_value"])
df = df[df["standard_units"].eq("nM")]

df.to_csv(OUTPUT_FILE, index=False)

print("Saved:", OUTPUT_FILE)
print("Rows:", len(df))
print(df.head())

In [ ]:
import pandas as pd
import numpy as np

# Load the file
df = pd.read_csv("egfr_nsclc_only.csv")
df.columns = df.columns.str.strip().str.lower()

# Clean string columns
string_cols = [
    "molecule_chembl_id", "assay_chembl_id", "document_chembl_id",
    "standard_type", "standard_units", "type", "units",
    "assay_description", "assay_type", "assay_organism",
    "doc_title", "doc_abstract"
]
for col in string_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

# Convert numeric columns
for col in ["standard_value", "value"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Keep only IC50 rows with numeric values
clean = df.copy()
if "standard_type" in clean.columns:
    clean = clean[clean["standard_type"].str.upper() == "IC50"].copy()

clean = clean.dropna(subset=["molecule_chembl_id", "standard_value"]).copy()

# Count missing values
missing_counts = clean.isna().sum()

# Count duplicates by molecule_chembl_id
dup_counts = clean["molecule_chembl_id"].value_counts()
duplicate_molecule_ids = int((dup_counts > 1).sum())
duplicate_extra_rows = int(dup_counts[dup_counts > 1].sum() - (dup_counts > 1).sum())

# Aggregate to one row per molecule
unique_df = (
    clean.groupby("molecule_chembl_id", as_index=False)
    .agg(
        ic50_nM=("standard_value", "median"),
        num_records=("standard_value", "count"),
        assay_chembl_id=("assay_chembl_id", "first"),
        document_chembl_id=("document_chembl_id", "first"),
        assay_description=("assay_description", "first"),
        assay_type=("assay_type", "first"),
        assay_organism=("assay_organism", "first"),
        doc_title=("doc_title", "first")
    )
)

# Convert IC50 nM to pIC50
unique_df["pic50"] = -np.log10(unique_df["ic50_nM"] * 1e-9)


# Save clean file
unique_df.to_csv("egfr_nsclc_unique_pIC50.csv", index=False)

# Save summary file
summary = pd.DataFrame([{
    "total_rows_original": len(df),
    "rows_after_ic50_filter": len(clean),
    "unique_molecule_chembl_id": unique_df["molecule_chembl_id"].nunique(),
    "duplicate_molecule_ids": duplicate_molecule_ids,
    "duplicate_extra_rows": duplicate_extra_rows,
    "missing_molecule_chembl_id": int(df["molecule_chembl_id"].isna().sum()) if "molecule_chembl_id" in df.columns else 0,
    "missing_standard_value": int(df["standard_value"].isna().sum()) if "standard_value" in df.columns else 0,
    "rows_in_clean_output": len(unique_df)
}])

summary.to_csv("egfr_nsclc_summary.csv", index=False)

# Print exact counts
print("Summary")
print(summary.to_string(index=False))

print("\nMissing values by column")
print(missing_counts[missing_counts > 0].sort_values(ascending=False))

print("\nFirst 10 rows of clean dataset")
print(unique_df.head(10).to_string(index=False))

In [ ]:
INPUT_FILE = "egfr_nsclc_unique_pIC50.csv"
OUTPUT_FILE = "egfr_nsclc_unique_pIC50_clean.csv"

df = pd.read_csv(INPUT_FILE)

keep_cols = [
    "molecule_chembl_id",
    "ic50_nM",
    "pic50"
]

keep_cols = [c for c in keep_cols if c in df.columns]
df = df[keep_cols]

df.to_csv(OUTPUT_FILE, index=False)

print("Saved:", OUTPUT_FILE)
print("Columns kept:", keep_cols)
print("Rows:", len(df))

In [ ]:
from chembl_webresource_client.new_client import new_client
from rdkit import Chem
from rdkit.Chem import inchi

INPUT_FILE = "egfr_nsclc_unique_pIC50_clean.csv"
OUTPUT_FILE = "egfr_nsclc_unique_pIC50_smiles_inchi.csv"

molecule = new_client.molecule
df = pd.read_csv(INPUT_FILE)

def smiles_to_inchi(smiles):
    if pd.isna(smiles):
        return None
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None
    try:
        return inchi.MolToInchi(mol, options='-SNon', logLevel=None)
    except Exception:
        return None

smiles_list, inchi_list = [], []

for mol_id in df["molecule_chembl_id"].astype(str):
    try:
        m = molecule.get(mol_id)
        s = (m.get("molecule_structures") or {}).get("canonical_smiles")
        smiles_list.append(s)
        inchi_list.append(smiles_to_inchi(s))
    except Exception:
        smiles_list.append(None)
        inchi_list.append(None)

df["smiles"] = smiles_list
df["inchi"] = inchi_list

df = df.dropna(subset=["smiles", "inchi"])
df.to_csv(OUTPUT_FILE, index=False)

print("Saved:", OUTPUT_FILE)
print("Rows:", len(df))
print(df[["molecule_chembl_id", "smiles", "inchi"]].head())

In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem

INPUT_FILE = "egfr_nsclc_unique_pIC50_smiles_inchi.csv"
OUTPUT_ATOMS = "atom_nodes.csv"
OUTPUT_BONDS = "bond_edges.csv"
OUTPUT_CONFORMERS = "conformers_3d.csv"

df = pd.read_csv(INPUT_FILE)
smiles_col = "smiles" if "smiles" in df.columns else "canonical_smiles"

atom_rows = []
bond_rows = []
conf_rows = []

for row_idx, row in df.iterrows():
    mol_id = row.get("molecule_chembl_id", f"mol_{row_idx}")
    smiles = row.get(smiles_col)

    if pd.isna(smiles):
        continue

    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        continue

    mol = Chem.AddHs(mol)

    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    res = AllChem.EmbedMolecule(mol, params)
    if res != 0:
        continue

    AllChem.UFFOptimizeMolecule(mol)

    conf = mol.GetConformer()

    for atom in mol.GetAtoms():
        atom_rows.append({
            "molecule_id": mol_id,
            "atom_idx": atom.GetIdx(),
            "atomic_num": atom.GetAtomicNum(),
            "symbol": atom.GetSymbol(),
            "formal_charge": atom.GetFormalCharge(),
            "degree": atom.GetDegree(),
            "num_hs": atom.GetTotalNumHs(),
            "is_aromatic": atom.GetIsAromatic(),
            "is_in_ring": atom.IsInRing()
        })

        pos = conf.GetAtomPosition(atom.GetIdx())
        conf_rows.append({
            "molecule_id": mol_id,
            "atom_idx": atom.GetIdx(),
            "x": pos.x,
            "y": pos.y,
            "z": pos.z
        })

    for bond in mol.GetBonds():
        bond_rows.append({
            "molecule_id": mol_id,
            "begin_atom_idx": bond.GetBeginAtomIdx(),
            "end_atom_idx": bond.GetEndAtomIdx(),
            "bond_type": str(bond.GetBondType()),
            "is_conjugated": bond.GetIsConjugated(),
            "is_in_ring": bond.IsInRing()
        })

atoms_df = pd.DataFrame(atom_rows)
bonds_df = pd.DataFrame(bond_rows)
confs_df = pd.DataFrame(conf_rows)

atoms_df.to_csv(OUTPUT_ATOMS, index=False)
bonds_df.to_csv(OUTPUT_BONDS, index=False)
confs_df.to_csv(OUTPUT_CONFORMERS, index=False)

print("Saved:", OUTPUT_ATOMS, OUTPUT_BONDS, OUTPUT_CONFORMERS)
print("Atoms:", len(atoms_df), "Bonds:", len(bonds_df), "Coords:", len(confs_df))

In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, Crippen, Lipinski

INPUT_FILE = "egfr_nsclc_unique_pIC50_smiles_inchi.csv"
DESCRIPTOR_FILE = "descriptors.csv"

df = pd.read_csv(INPUT_FILE)

smiles_col = None
for c in ["smiles", "canonical_smiles"]:
    if c in df.columns:
        smiles_col = c
        break

if smiles_col is None:
    raise ValueError("No SMILES column found. Expected 'smiles' or 'canonical_smiles'.")

rows = []

for i, row in df.iterrows():
    mol_id = row["molecule_chembl_id"] if "molecule_chembl_id" in df.columns else f"mol_{i}"
    smiles = row[smiles_col]

    if pd.isna(smiles):
        continue

    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        continue

    rows.append({
        "molecule_id": mol_id,
        "smiles": str(smiles),
        "mol_wt": Descriptors.MolWt(mol),
        "tpsa": rdMolDescriptors.CalcTPSA(mol),
        "logp": Crippen.MolLogP(mol),
        "hba": Lipinski.NumHAcceptors(mol),
        "hbd": Lipinski.NumHDonors(mol),
        "rotatable_bonds": Lipinski.NumRotatableBonds(mol),
        "ring_count": rdMolDescriptors.CalcNumRings(mol),
        "fraction_csp3": rdMolDescriptors.CalcFractionCSP3(mol)
    })

desc_df = pd.DataFrame(rows)
desc_df.to_csv(DESCRIPTOR_FILE, index=False)

print("Saved:", DESCRIPTOR_FILE)
print("Rows:", len(desc_df))
print("Columns:", len(desc_df.columns))

In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem, DataStructs
from rdkit.Chem import MACCSkeys
from rdkit.Chem import rdFingerprintGenerator

DESCRIPTOR_FILE = "descriptors.csv"
OUTPUT_FILE = "descriptors_with_fingerprints.csv"

df = pd.read_csv(DESCRIPTOR_FILE)

morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

rows = []

for _, row in df.iterrows():
    smiles = row["smiles"]
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        continue

    out = row.to_dict()

    morgan_fp = morgan_gen.GetFingerprint(mol)
    morgan_arr = np.zeros((2048,), dtype=int)
    DataStructs.ConvertToNumpyArray(morgan_fp, morgan_arr)
    for j, bit in enumerate(morgan_arr):
        out[f"morgan_{j}"] = int(bit)

    maccs_fp = MACCSkeys.GenMACCSKeys(mol)
    maccs_arr = np.zeros((167,), dtype=int)
    DataStructs.ConvertToNumpyArray(maccs_fp, maccs_arr)
    for j in range(1, 167):
        out[f"maccs_{j}"] = int(maccs_arr[j])

    rows.append(out)

final_df = pd.DataFrame(rows)
final_df.to_csv(OUTPUT_FILE, index=False)

print("Saved:", OUTPUT_FILE)
print("Rows:", len(final_df))
print("Columns:", len(final_df.columns))

In [ ]:
import pandas as pd
import numpy as np

activity = pd.read_csv("egfr_nsclc_unique_pIC50_clean.csv")
features = pd.read_csv("descriptors_with_fingerprints.csv")

# Make sure pIC50 is numeric
activity["pic50"] = pd.to_numeric(activity["pic50"], errors="coerce")
activity = activity.dropna(subset=["pic50", "molecule_chembl_id"])

# Define active from pIC50
# pIC50 >= 7 means IC50 <= 1000 nM
activity["active"] = (activity["pic50"] >= 7).astype(int)

# Keep only needed columns and remove duplicate molecules
activity = activity[["molecule_chembl_id", "active"]].drop_duplicates()

# Standardize merge key
features = features.rename(columns={"molecule_id": "molecule_chembl_id"})

# Merge feature table with labels
df = features.merge(activity, on="molecule_chembl_id", how="inner")

# Save
df.to_csv("module4_ready.csv", index=False)

print(df["active"].value_counts())
print("Saved: module4_ready.csv")

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

INPUT_FILE = "module4_ready.csv"   # change if needed

df = pd.read_csv(INPUT_FILE)

fp_cols = [c for c in df.columns if c.startswith("morgan_")]
if not fp_cols:
    raise ValueError("No Morgan fingerprint columns found in the input file.")

target_col = None
for c in ["active", "label", "activity", "is_active"]:
    if c in df.columns:
        target_col = c
        break

if target_col is None:
    raise ValueError("No label column found. Add one of: active, label, activity, is_active")

X = df[fp_cols]
y = df[target_col].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

test_ids = df.loc[X_test.index, "molecule_id"].values if "molecule_id" in df.columns else X_test.index
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Positive class ratio:", y.mean())

In [ ]:
import pandas as pd
import warnings
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore", message="No further splits with positive gain")

main_models = {
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1
    )
}

comparison_models = {
    "LightGBM": LGBMClassifier(
        n_estimators=200,
        learning_rate=0.05,
        num_leaves=15,
        min_child_samples=10,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )
}

results = []
pred_df = pd.DataFrame({
    "molecule_id": test_ids,
    "true_label": y_test.values
})

for name, model in main_models.items():
    model.fit(X_train, y_train)
    prob = model.predict_proba(X_test)[:, 1]
    pred = model.predict(X_test)

    results.append({
        "model": name,
        "role": "main",
        "roc_auc": roc_auc_score(y_test, prob),
        "accuracy": accuracy_score(y_test, pred)
    })

    pred_df[f"{name}_prob_active"] = prob
    pred_df[f"{name}_pred"] = pred

    print(f"\n=== {name} ===")
    print("ROC-AUC:", roc_auc_score(y_test, prob))
    print("Accuracy:", accuracy_score(y_test, pred))
    print("Confusion matrix:\n", confusion_matrix(y_test, pred))
    print(classification_report(y_test, pred))

for name, model in comparison_models.items():
    model.fit(X_train, y_train)
    prob = model.predict_proba(X_test)[:, 1]
    pred = model.predict(X_test)

    results.append({
        "model": name,
        "role": "comparison",
        "roc_auc": roc_auc_score(y_test, prob),
        "accuracy": accuracy_score(y_test, pred)
    })

    pred_df[f"{name}_prob_active"] = prob
    pred_df[f"{name}_pred"] = pred

    print(f"\n=== {name} ===")
    print("ROC-AUC:", roc_auc_score(y_test, prob))
    print("Accuracy:", accuracy_score(y_test, pred))
    print("Confusion matrix:\n", confusion_matrix(y_test, pred))
    print(classification_report(y_test, pred))

results_df = pd.DataFrame(results)
main_results_df = results_df[results_df["role"] == "main"].sort_values("roc_auc", ascending=False)
comparison_results_df = results_df[results_df["role"] == "comparison"].sort_values("roc_auc", ascending=False)

main_results_df.to_csv("module4_main_model_metrics.csv", index=False)
comparison_results_df.to_csv("module4_comparison_model_metrics.csv", index=False)
pred_df.to_csv("module4_test_predictions.csv", index=False)

best_main_model = main_results_df.iloc[0]["model"]
print("\nBest main model:", best_main_model)
print("\nSaved: module4_main_model_metrics.csv")
print("Saved: module4_comparison_model_metrics.csv")
print("Saved: module4_test_predictions.csv")

In [ ]:
import pandas as pd
import torch
from rdkit import Chem
from torch_geometric.data import Data

INPUT_FILE = "egfr_nsclc_unique_pIC50_smiles_inchi.csv"   # must contain smiles and pIC50

df = pd.read_csv(INPUT_FILE)

if "smiles" not in df.columns:
    raise ValueError("No 'smiles' column found.")
if "pic50" not in df.columns:
    raise ValueError("No 'pic50' column found.")

df["pic50"] = pd.to_numeric(df["pic50"], errors="coerce")
df = df.dropna(subset=["smiles", "pic50"]).copy()

def atom_features(atom):
    return [
        atom.GetAtomicNum(),
        atom.GetDegree(),
        atom.GetFormalCharge(),
        atom.GetTotalNumHs(),
        int(atom.GetIsAromatic()),
        int(atom.IsInRing())
    ]

def bond_features(bond):
    return [
        float(bond.GetBondTypeAsDouble()),
        int(bond.GetIsAromatic()),
        int(bond.GetIsConjugated()),
        int(bond.IsInRing())
    ]

def mol_to_graph(smiles, y_value, mol_id=None):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None

    x = torch.tensor([atom_features(atom) for atom in mol.GetAtoms()], dtype=torch.float)

    edge_indices = []
    edge_attrs = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        bf = bond_features(bond)

        edge_indices.append([i, j])
        edge_indices.append([j, i])
        edge_attrs.append(bf)
        edge_attrs.append(bf)

    if len(edge_indices) == 0:
        return None

    edge_index = torch.tensor(edge_indices, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attrs, dtype=torch.float)
    y = torch.tensor([float(y_value)], dtype=torch.float)

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)
    data.smiles = str(smiles)
    if mol_id is not None:
        data.molecule_id = str(mol_id)
    return data

graphs = []
for _, row in df.iterrows():
    mol_id = row["molecule_chembl_id"] if "molecule_chembl_id" in df.columns else None
    g = mol_to_graph(row["smiles"], row["pic50"], mol_id)
    if g is not None:
        graphs.append(g)

print("Graphs:", len(graphs))
print("Node feature dimension:", graphs[0].num_node_features)
print("Example graph:", graphs[0])

In [ ]:
from sklearn.model_selection import train_test_split
from torch_geometric.loader import DataLoader

train_graphs, test_graphs = train_test_split(
    graphs,
    test_size=0.2,
    random_state=42
)

train_loader = DataLoader(train_graphs, batch_size=32, shuffle=True)
test_loader = DataLoader(test_graphs, batch_size=32, shuffle=False)

print("Train graphs:", len(train_graphs))
print("Test graphs:", len(test_graphs))

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool

class GCNRegressor(nn.Module):
    def __init__(self, num_node_features, hidden_dim=64):
        super().__init__()
        self.conv1 = GCNConv(num_node_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim)
        self.lin1 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.lin2 = nn.Linear(hidden_dim // 2, 1)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = F.relu(self.conv3(x, edge_index))
        x = global_mean_pool(x, batch)
        x = F.relu(self.lin1(x))
        x = self.lin2(x)
        return x.view(-1)

In [ ]:
import torch
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = GCNRegressor(num_node_features=graphs[0].num_node_features).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

def train_one_epoch():
    model.train()
    total_loss = 0.0

    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        pred = model(batch)
        loss = criterion(pred, batch.y.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs

    return total_loss / len(train_loader.dataset)

def evaluate(loader):
    model.eval()
    total_loss = 0.0
    preds, targets = [], []

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            pred = model(batch)
            loss = criterion(pred, batch.y.view(-1))
            total_loss += loss.item() * batch.num_graphs
            preds.extend(pred.cpu().numpy())
            targets.extend(batch.y.view(-1).cpu().numpy())

    preds = np.array(preds)
    targets = np.array(targets)
    rmse = np.sqrt(mean_squared_error(targets, preds))
    r2 = r2_score(targets, preds)

    return total_loss / len(loader.dataset), rmse, r2, preds, targets

for epoch in range(1, 51):
    train_loss = train_one_epoch()
    test_loss, rmse, r2, _, _ = evaluate(test_loader)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Test Loss: {test_loss:.4f} | RMSE: {rmse:.4f} | R2: {r2:.4f}")

In [ ]:
model.eval()
rows = []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        pred = model(batch).cpu().numpy()
        true = batch.y.view(-1).cpu().numpy()

        mol_ids = batch.molecule_id if hasattr(batch, "molecule_id") else [None] * len(pred)
        smiles_list = batch.smiles if hasattr(batch, "smiles") else [None] * len(pred)

        for i in range(len(pred)):
            rows.append({
                "molecule_id": mol_ids[i] if mol_ids is not None else None,
                "smiles": smiles_list[i] if smiles_list is not None else None,
                "true_pIC50": float(true[i]),
                "predicted_pIC50": float(pred[i])
            })

pred_df = pd.DataFrame(rows)
pred_df.to_csv("module5_GCN_test_predictions.csv", index=False)

metrics_df = pd.DataFrame([{
    "test_loss": test_loss,
    "rmse": rmse,
    "r2": r2
}])
metrics_df.to_csv("module5_GCN_metrics.csv", index=False)

print("Saved: module5_GCN_test_predictions.csv")
print("Saved: module5_GCN_metrics.csv")
print(metrics_df)

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from rdkit import Chem
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import NNConv, global_mean_pool

INPUT_FILE = "egfr_nsclc_unique_pIC50_smiles_inchi.csv"

df = pd.read_csv(INPUT_FILE)
df["pic50"] = pd.to_numeric(df["pic50"], errors="coerce")
df = df.dropna(subset=["smiles", "pic50"]).copy()

def atom_features(atom):
    return [
        atom.GetAtomicNum(),
        atom.GetDegree(),
        atom.GetFormalCharge(),
        atom.GetTotalNumHs(),
        int(atom.GetIsAromatic()),
        int(atom.IsInRing())
    ]

def bond_features(bond):
    return [
        float(bond.GetBondTypeAsDouble()),
        int(bond.GetIsAromatic()),
        int(bond.GetIsConjugated()),
        int(bond.IsInRing())
    ]

def mol_to_graph(smiles, y_value, mol_id=None):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None

    x = torch.tensor([atom_features(atom) for atom in mol.GetAtoms()], dtype=torch.float)

    edge_indices = []
    edge_attrs = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        bf = bond_features(bond)
        edge_indices.append([i, j])
        edge_indices.append([j, i])
        edge_attrs.append(bf)
        edge_attrs.append(bf)

    if len(edge_indices) == 0:
        return None

    edge_index = torch.tensor(edge_indices, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attrs, dtype=torch.float)
    y = torch.tensor([float(y_value)], dtype=torch.float)

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)
    data.smiles = str(smiles)
    data.molecule_id = str(mol_id) if mol_id is not None else "NA"
    return data

graphs = []
for _, row in df.iterrows():
    mol_id = row["molecule_chembl_id"] if "molecule_chembl_id" in df.columns else None
    g = mol_to_graph(row["smiles"], row["pic50"], mol_id)
    if g is not None:
        graphs.append(g)

train_graphs, test_graphs = train_test_split(graphs, test_size=0.2, random_state=42)
train_loader = DataLoader(train_graphs, batch_size=32, shuffle=True)
test_loader = DataLoader(test_graphs, batch_size=32, shuffle=False)

class MPNNRegressor(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim=64):
        super().__init__()
        self.node_proj = nn.Linear(node_dim, hidden_dim)

        edge_network1 = nn.Sequential(nn.Linear(edge_dim, hidden_dim * hidden_dim))
        edge_network2 = nn.Sequential(nn.Linear(edge_dim, hidden_dim * hidden_dim))

        self.conv1 = NNConv(hidden_dim, hidden_dim, edge_network1, aggr="mean")
        self.conv2 = NNConv(hidden_dim, hidden_dim, edge_network2, aggr="mean")

        self.readout = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, data):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        x = F.relu(self.node_proj(x))
        x = F.relu(self.conv1(x, edge_index, edge_attr))
        x = F.relu(self.conv2(x, edge_index, edge_attr))
        x = global_mean_pool(x, batch)
        return self.readout(x).view(-1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MPNNRegressor(
    node_dim=graphs[0].num_node_features,
    edge_dim=graphs[0].edge_attr.shape[1],
    hidden_dim=64
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

def train_one_epoch():
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        pred = model(batch)
        loss = criterion(pred, batch.y.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
    return total_loss / len(train_loader.dataset)

def evaluate(loader):
    model.eval()
    total_loss = 0.0
    preds, targets = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            pred = model(batch)
            loss = criterion(pred, batch.y.view(-1))
            total_loss += loss.item() * batch.num_graphs
            preds.extend(pred.cpu().numpy())
            targets.extend(batch.y.view(-1).cpu().numpy())
    preds = np.array(preds)
    targets = np.array(targets)
    rmse = np.sqrt(mean_squared_error(targets, preds))
    r2 = r2_score(targets, preds)
    return total_loss / len(loader.dataset), rmse, r2

for epoch in range(1, 51):
    train_loss = train_one_epoch()
    test_loss, rmse, r2 = evaluate(test_loader)
    if epoch == 1 or epoch % 5 == 0:
        print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Test Loss: {test_loss:.4f} | RMSE: {rmse:.4f} | R2: {r2:.4f}")

In [ ]:
test_loss, rmse, r2 = evaluate(test_loader)

model.eval()
rows = []

with torch.no_grad():
    for batch in test_loader:
        batch_cpu = batch
        batch = batch.to(device)

        pred = model(batch).cpu().numpy()
        true = batch.y.view(-1).cpu().numpy()

        mol_ids = batch_cpu.molecule_id if hasattr(batch_cpu, "molecule_id") else ["NA"] * len(pred)
        smiles_list = batch_cpu.smiles if hasattr(batch_cpu, "smiles") else ["NA"] * len(pred)

        for i in range(len(pred)):
            rows.append({
                "molecule_id": mol_ids[i],
                "smiles": smiles_list[i],
                "true_pIC50": float(true[i]),
                "predicted_pIC50": float(pred[i])
            })

pred_df = pd.DataFrame(rows)
pred_df.to_csv("module5_MPNN_test_predictions.csv", index=False)

metrics_df = pd.DataFrame([{
    "test_loss": float(test_loss),
    "rmse": float(rmse),
    "r2": float(r2)
}])
metrics_df.to_csv("module5_MPNN_metrics.csv", index=False)

torch.save(model.state_dict(), "module5_mpnn_model.pt")

print("Saved: module5_MPNN_test_predictions.csv")
print("Saved: module5_MPNN_metrics.csv")
print("Saved: module5_mpnn_model.pt")
print(metrics_df)

In [ ]:
!git clone https://github.com/swansonk14/admet_ai.git
!pip install admet-ai

In [ ]:
import pandas as pd
from admet_ai import ADMETModel

# Load input molecules
input_df = pd.read_csv("egfr_nsclc_unique_pIC50_smiles_inchi.csv")
input_df.columns = input_df.columns.str.strip().str.lower()

# Keep only valid rows
input_df = input_df.dropna(subset=["molecule_chembl_id", "smiles"]).copy()
input_df["smiles"] = input_df["smiles"].astype(str).str.strip()

smiles_list = input_df["smiles"].tolist()

# Predict ADMET
model = ADMETModel()
all_preds = model.predict(smiles=smiles_list)

# all_preds is a DataFrame indexed by SMILES when input is a list
all_preds = all_preds.reset_index().rename(columns={"index": "smiles"})

# Add identifiers back by merging on SMILES
out = input_df[["molecule_chembl_id", "smiles", "pic50"]].merge(
    all_preds,
    on="smiles",
    how="inner"
)

out.to_csv("module6_admet.csv", index=False)
print("Saved module6_admet.csv")

In [ ]:
import pandas as pd
import numpy as np

# Load raw Module 6 ADMET output
df = pd.read_csv("module6_admet.csv")
df.columns = df.columns.str.strip().str.lower()

def quality_score(series):
    s = pd.to_numeric(series, errors="coerce")
    non_null_ratio = s.notna().mean()
    unique_ratio = s.nunique(dropna=True) / max(len(s.dropna()), 1)
    std_val = s.std(skipna=True)
    std_score = 0 if pd.isna(std_val) else min(std_val, 1e6) / (min(std_val, 1e6) + 1)
    return 0.6 * non_null_ratio + 0.25 * unique_ratio + 0.15 * std_score

def choose_best_numeric_column(data, candidates):
    available = [c for c in candidates if c in data.columns]
    if not available:
        return None
    scores = {c: quality_score(data[c]) for c in available}
    return max(scores, key=scores.get)

def label_3way(series, higher_is_worse=True):
    s = pd.to_numeric(series, errors="coerce")
    if s.notna().sum() == 0:
        return pd.Series(["Unknown"] * len(s), index=s.index)

    q1, q2 = s.quantile([0.33, 0.66]).values

    if higher_is_worse:
        return pd.Series(
            np.where(s <= q1, "Low", np.where(s <= q2, "Medium", "High")),
            index=s.index
        )
    else:
        return pd.Series(
            np.where(s <= q1, "High", np.where(s <= q2, "Medium", "Low")),
            index=s.index
        )

# Pick best toxicity and clearance columns by data quality
tox_candidates = ["ames", "clintox", "dili", "herg"]
clearance_candidates = ["clearance_hepatocyte_az", "clearance_microsome_az"]

best_tox_col = choose_best_numeric_column(df, tox_candidates)
best_clearance_col = choose_best_numeric_column(df, clearance_candidates)

# Build final output
out = pd.DataFrame()
out["molecule_chembl_id"] = df["molecule_chembl_id"] if "molecule_chembl_id" in df.columns else np.nan
out["smiles"] = df["smiles"] if "smiles" in df.columns else np.nan
out["pic50"] = pd.to_numeric(df["pic50"], errors="coerce").round(2) if "pic50" in df.columns else np.nan

# Solubility stays numeric
if "solubility_aqsoldb" in df.columns:
    out["solubility"] = pd.to_numeric(df["solubility_aqsoldb"], errors="coerce").round(2)
else:
    out["solubility"] = np.nan

# Toxicity: higher score = worse toxicity
if best_tox_col is not None:
    out["toxicity"] = label_3way(df[best_tox_col], higher_is_worse=True)
else:
    out["toxicity"] = "Unknown"

# Clearance: better project-wise to report low/medium/high from selected column
if best_clearance_col is not None:
    out["clearance"] = label_3way(df[best_clearance_col], higher_is_worse=False)
else:
    out["clearance"] = "Unknown"

# Half-life shown in hours
if "half_life_obach" in df.columns:
    hl = pd.to_numeric(df["half_life_obach"], errors="coerce")
    out["half_life"] = np.where(hl.notna(), hl.round(2).astype(str) + " hours", "Unknown")
else:
    out["half_life"] = "Unknown"

# Optional tracking columns so you know what was chosen
out["toxicity_source"] = best_tox_col if best_tox_col is not None else "None"
out["clearance_source"] = best_clearance_col if best_clearance_col is not None else "None"

# Save final clean Module 6 output
out.to_csv("module6_output.csv", index=False)

print("Saved module6_output.csv")
print("Best toxicity column:", best_tox_col)
print("Best clearance column:", best_clearance_col)
print(out.head(10).to_string(index=False))

In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, Crippen
from rdkit.Contrib.SA_Score import sascorer


# Load data
df = pd.read_csv("module6_admet.csv")
df.columns = df.columns.str.strip().str.lower()

for col in ["pic50", "solubility_aqsoldb", "ames", "clintox", "dili", "herg"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

def minmax(s):
    s = pd.to_numeric(s, errors="coerce")
    if s.notna().sum() == 0:
        return pd.Series(0.5, index=s.index)
    mn, mx = s.min(), s.max()
    if pd.isna(mn) or pd.isna(mx) or mn == mx:
        return pd.Series(0.5, index=s.index)
    return (s - mn) / (mx - mn)

def sa_score(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.nan
    return sascorer.calculateScore(mol)

# Raw columns
tracking = df[["molecule_chembl_id", "smiles", "pic50"]].copy()

if "solubility_aqsoldb" in df.columns:
    tracking["solubility_raw"] = df["solubility_aqsoldb"]
else:
    tracking["solubility_raw"] = np.nan

tox_candidates = ["ames", "clintox", "dili", "herg"]
tox_col = next((c for c in tox_candidates if c in df.columns), None)
if tox_col is not None:
    tracking["toxicity_raw"] = df[tox_col]
    tracking["toxicity_source"] = tox_col
else:
    tracking["toxicity_raw"] = np.nan
    tracking["toxicity_source"] = None

if "smiles" in df.columns:
    tracking["sa_raw"] = df["smiles"].apply(sa_score)
else:
    tracking["sa_raw"] = np.nan

# Normalized component scores
tracking["potency_score"] = minmax(tracking["pic50"])
tracking["solubility_score"] = minmax(tracking["solubility_raw"])
tracking["toxicity_score"] = 1 - minmax(tracking["toxicity_raw"]) if tracking["toxicity_raw"].notna().any() else 0.5
tracking["sa_score"] = 1 - minmax(tracking["sa_raw"]) if tracking["sa_raw"].notna().any() else 0.5

# Final score
tracking["final_score"] = (
    0.4 * tracking["potency_score"].fillna(0.5) +
    0.2 * tracking["solubility_score"].fillna(0.5) +
    0.2 * tracking["toxicity_score"].fillna(0.5) +
    0.2 * tracking["sa_score"].fillna(0.5)
)

# Save score tracking table
tracking.to_csv("module7_score_tracking.csv", index=False)

# Ranked output
ranked = tracking.sort_values("final_score", ascending=False).reset_index(drop=True)
ranked.to_csv("module7_ranked_molecules.csv", index=False)

print("Saved module7_score_tracking.csv")
print("Saved module7_ranked_molecules.csv")
print(ranked.head(10).to_string(index=False))

In [ ]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("egfr_ic50_full.csv")
df.columns = df.columns.str.lower().str.strip()

# Keep only IC50 rows
df = df[df["standard_type"].astype(str).str.upper() == "IC50"].copy()
df["standard_value"] = pd.to_numeric(df["standard_value"], errors="coerce")

# Convert to nM if needed
def to_nm(row):
    unit = str(row["standard_units"]).lower()
    val = row["standard_value"]
    if pd.isna(val):
        return np.nan
    if unit == "nm":
        return val
    if unit in ["um", "µm", "μm"]:
        return val * 1000
    return np.nan

df["ic50_nm"] = df.apply(to_nm, axis=1)
df["pic50"] = np.where(df["ic50_nm"] > 0, 9 - np.log10(df["ic50_nm"]), np.nan)

# 1. Missing Data
missing_data = pd.DataFrame({
    "column": df.columns,
    "missing_count": [df[c].isna().sum() for c in df.columns],
    "missing_percent": [df[c].isna().mean() * 100 for c in df.columns]
})
missing_data.to_csv("module10_missing_data_audit.csv", index=False)

# 2. Duplicate Molecules
duplicate_molecules = df[df.duplicated(subset=["molecule_chembl_id"], keep=False)].copy()
duplicate_molecules.to_csv("module10_duplicate_molecules_audit.csv", index=False)

# 3. Assay Noise
assay_noise = df.dropna(subset=["ic50_nm"]).groupby("molecule_chembl_id", as_index=False).agg(
    n_measurements=("ic50_nm", "size"),
    min_ic50_nm=("ic50_nm", "min"),
    max_ic50_nm=("ic50_nm", "max"),
    median_ic50_nm=("ic50_nm", "median"),
    median_pic50=("pic50", "median")
)

assay_noise = assay_noise[assay_noise["n_measurements"] > 1].copy()
assay_noise["fold_change"] = assay_noise["max_ic50_nm"] / assay_noise["min_ic50_nm"]
assay_noise["noise_flag"] = np.where(
    assay_noise["fold_change"] >= 10, "high_noise",
    np.where(assay_noise["fold_change"] >= 3, "moderate_noise", "low_noise")
)
assay_noise.to_csv("module10_assay_noise_audit.csv", index=False)

print("Created:")
print("module10_missing_data_audit.csv")
print("module10_duplicate_molecules_audit.csv")
print("module10_assay_noise_audit.csv")

In [ ]:
import os
import pandas as pd
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

INPUT_FILE = "egfr_ic50_full.csv"
OUTPUT_FILE = "module10_molecule_smiles_lookup.csv"
MAX_WORKERS = 12

# Load dataset
df = pd.read_csv(INPUT_FILE)
df.columns = df.columns.str.lower().str.strip()

all_ids = (
    df["molecule_chembl_id"]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .tolist()
)

# Resume support
if os.path.exists(OUTPUT_FILE):
    existing = pd.read_csv(OUTPUT_FILE)
    existing.columns = existing.columns.str.lower().str.strip()
    done_ids = set(existing["molecule_chembl_id"].dropna().astype(str))
else:
    existing = pd.DataFrame(columns=["molecule_chembl_id", "canonical_smiles"])
    done_ids = set()

remaining_ids = [cid for cid in all_ids if cid not in done_ids]

print(f"Total unique IDs   : {len(all_ids)}")
print(f"Already completed  : {len(done_ids)}")
print(f"Remaining to fetch : {len(remaining_ids)}")

def make_session():
    session = requests.Session()
    retry = Retry(
        total=5,
        connect=5,
        read=5,
        backoff_factor=0.5,
        status_forcelist=[408, 429, 500, 502, 503, 504],
        allowed_methods=["GET"]
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    return session

def fetch_smiles(chembl_id):
    session = make_session()
    url = f"https://www.ebi.ac.uk/chembl/api/data/molecule/{chembl_id}.json"
    try:
        r = session.get(url, timeout=20)
        if r.status_code == 200:
            data = r.json()
            mol_struct = data.get("molecule_structures")
            if mol_struct:
                return {
                    "molecule_chembl_id": chembl_id,
                    "canonical_smiles": mol_struct.get("canonical_smiles")
                }
    except:
        pass
    return {
        "molecule_chembl_id": chembl_id,
        "canonical_smiles": None
    }

new_rows = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(fetch_smiles, cid): cid for cid in remaining_ids}

    for i, future in enumerate(as_completed(futures), start=1):
        result = future.result()
        new_rows.append(result)

        if i % 50 == 0:
            print(f"Completed {i}/{len(remaining_ids)}")

# Save final result
new_df = pd.DataFrame(new_rows)
final_df = pd.concat([existing, new_df], ignore_index=True)
final_df = final_df.drop_duplicates(subset=["molecule_chembl_id"], keep="first")
final_df.to_csv(OUTPUT_FILE, index=False)

print(f"Done. Created/updated: {OUTPUT_FILE}")
print(final_df.head())

In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator

ACTIVITY_FILE = "egfr_ic50_full.csv"
SMILES_FILE = "module10_molecule_smiles_lookup.csv"
OUTPUT_FILE = "module10_activity_cliffs_audit.csv"
SUMMARY_FILE = "module10_activity_cliff_summary.csv"

df = pd.read_csv(ACTIVITY_FILE)
df.columns = df.columns.str.lower().str.strip()

df = df[df["standard_type"].astype(str).str.upper() == "IC50"].copy()
df["standard_value"] = pd.to_numeric(df["standard_value"], errors="coerce")
df["ic50_nm"] = df["standard_value"]
df["pic50"] = np.where(df["ic50_nm"] > 0, 9 - np.log10(df["ic50_nm"]), np.nan)

mol_activity = df.groupby("molecule_chembl_id", as_index=False).agg(
    n_assays=("ic50_nm", "size"),
    median_ic50_nm=("ic50_nm", "median"),
    median_pic50=("pic50", "median")
)

def label_activity(ic50_nm):
    if pd.isna(ic50_nm):
        return "unknown"
    elif ic50_nm <= 100:
        return "very_active"
    elif ic50_nm <= 1000:
        return "active"
    elif ic50_nm <= 10000:
        return "moderate"
    else:
        return "inactive"

mol_activity["activity_label"] = mol_activity["median_ic50_nm"].apply(label_activity)

smiles_df = pd.read_csv(SMILES_FILE)
smiles_df.columns = smiles_df.columns.str.lower().str.strip()

mols = mol_activity.merge(smiles_df, on="molecule_chembl_id", how="left")
mols = mols.dropna(subset=["canonical_smiles", "median_pic50"]).copy()

morgan_gen = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048
)

fps = []
valid_rows = []

for _, row in mols.iterrows():
    mol = Chem.MolFromSmiles(row["canonical_smiles"])
    if mol is not None:
        fp = morgan_gen.GetFingerprint(mol)
        fps.append(fp)
        valid_rows.append(row)

mols = pd.DataFrame(valid_rows).reset_index(drop=True)

cliff_rows = []

for i in range(len(mols)):
    for j in range(i + 1, len(mols)):
        sim = DataStructs.TanimotoSimilarity(fps[i], fps[j])
        delta = abs(mols.loc[i, "median_pic50"] - mols.loc[j, "median_pic50"])

        if sim >= 0.85 and delta >= 2.0:
            row_i = mols.loc[i]
            row_j = mols.loc[j]

            cliff_rows.append({
                "molecule_a": row_i["molecule_chembl_id"],
                "molecule_b": row_j["molecule_chembl_id"],
                "smiles_a": row_i["canonical_smiles"],
                "smiles_b": row_j["canonical_smiles"],
                "similarity_tanimoto": round(sim, 4),
                "molecule_a_ic50_nm": round(float(row_i["median_ic50_nm"]), 4),
                "molecule_b_ic50_nm": round(float(row_j["median_ic50_nm"]), 4),
                "molecule_a_pic50": round(float(row_i["median_pic50"]), 4),
                "molecule_b_pic50": round(float(row_j["median_pic50"]), 4),
                "delta_pic50": round(float(delta), 4),
                "molecule_a_activity": row_i["activity_label"],
                "molecule_b_activity": row_j["activity_label"],
                "molecule_a_assay_count": int(row_i["n_assays"]),
                "molecule_b_assay_count": int(row_j["n_assays"]),
                "cliff_case": f"{row_i['activity_label']} vs {row_j['activity_label']}"
            })

activity_cliffs = pd.DataFrame(cliff_rows)

if not activity_cliffs.empty:
    activity_cliffs = activity_cliffs.sort_values(
        by=["delta_pic50", "similarity_tanimoto"],
        ascending=[False, False]
    ).reset_index(drop=True)

activity_cliffs.to_csv(OUTPUT_FILE, index=False)

summary = pd.DataFrame({
    "metric": [
        "total_molecules_with_smiles",
        "total_activity_cliffs"
    ],
    "value": [
        len(mols),
        len(activity_cliffs)
    ]
})

summary.to_csv(SUMMARY_FILE, index=False)

print(f"Created {OUTPUT_FILE}")
print(f"Created {SUMMARY_FILE}")

In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

INPUT_FILE = "egfr_ic50_full.csv"
SMILES_FILE = "module10_molecule_smiles_lookup.csv"
OUTPUT_FILE = "module11_validation_results.csv"

df = pd.read_csv(INPUT_FILE)
df.columns = df.columns.str.lower().str.strip()

df = df[df["standard_type"].astype(str).str.upper() == "IC50"].copy()
df["standard_value"] = pd.to_numeric(df["standard_value"], errors="coerce")
df = df.dropna(subset=["standard_value"]).copy()

df["ic50_nm"] = df["standard_value"]
df["pic50"] = np.where(df["ic50_nm"] > 0, 9 - np.log10(df["ic50_nm"]), np.nan)

mol_activity = df.groupby("molecule_chembl_id", as_index=False).agg(
    median_ic50_nm=("ic50_nm", "median"),
    median_pic50=("pic50", "median"),
    first_doc=("document_chembl_id", "first")
)
mol_activity["label"] = (mol_activity["median_ic50_nm"] <= 1000).astype(int)

smiles_df = pd.read_csv(SMILES_FILE)
smiles_df.columns = smiles_df.columns.str.lower().str.strip()

data = mol_activity.merge(smiles_df, on="molecule_chembl_id", how="left")
data = data.dropna(subset=["canonical_smiles"]).copy()

morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def mol_from_smiles(s):
    try:
        return Chem.MolFromSmiles(s)
    except:
        return None

def fp_to_array(mol):
    fp = morgan_gen.GetFingerprint(mol)
    arr = np.zeros((2048,), dtype=int)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

valid_rows = []
fp_rows = []

for _, row in data.iterrows():
    mol = mol_from_smiles(row["canonical_smiles"])
    if mol is not None:
        valid_rows.append(row)
        fp_rows.append(fp_to_array(mol))

data = pd.DataFrame(valid_rows).reset_index(drop=True)
X = np.vstack(fp_rows)
y = data["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf_random = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf_random.fit(X_train, y_train)
pred_random = rf_random.predict_proba(X_test)[:, 1]

random_auc = roc_auc_score(y_test, pred_random)
random_acc = accuracy_score(y_test, (pred_random >= 0.5).astype(int))

def scaffold_from_smiles(smiles):
    mol = mol_from_smiles(smiles)
    if mol is None:
        return None
    scaffold = MurckoScaffold.GetScaffoldForMol(mol)
    return Chem.MolToSmiles(scaffold) if scaffold is not None else None

data["scaffold"] = data["canonical_smiles"].apply(scaffold_from_smiles)
data = data.dropna(subset=["scaffold"]).copy()

scaffold_sizes = data.groupby("scaffold").size().sort_values(ascending=False)
target_test = int(0.2 * len(data))
test_scaffolds = []
count = 0

for scaffold, size in scaffold_sizes.items():
    if count < target_test:
        test_scaffolds.append(scaffold)
        count += size

scaffold_test_mask = data["scaffold"].isin(test_scaffolds)

train_sc = data.loc[~scaffold_test_mask].copy()
test_sc = data.loc[scaffold_test_mask].copy()

X_sc_train = np.vstack([
    fp_to_array(mol_from_smiles(s))
    for s in train_sc["canonical_smiles"]
    if mol_from_smiles(s) is not None
])
y_sc_train = train_sc["label"].values

X_sc_test = np.vstack([
    fp_to_array(mol_from_smiles(s))
    for s in test_sc["canonical_smiles"]
    if mol_from_smiles(s) is not None
])
y_sc_test = test_sc["label"].values

rf_scaffold = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf_scaffold.fit(X_sc_train, y_sc_train)
pred_scaffold = rf_scaffold.predict_proba(X_sc_test)[:, 1]

scaffold_auc = roc_auc_score(y_sc_test, pred_scaffold) if len(np.unique(y_sc_test)) > 1 else np.nan
scaffold_acc = accuracy_score(y_sc_test, (pred_scaffold >= 0.5).astype(int))

temp_data = data.sort_values("first_doc").reset_index(drop=True)
split_idx = int(0.8 * len(temp_data))

train_temp = temp_data.iloc[:split_idx].copy()
test_temp = temp_data.iloc[split_idx:].copy()

X_temp_train = np.vstack([
    fp_to_array(mol_from_smiles(s))
    for s in train_temp["canonical_smiles"]
    if mol_from_smiles(s) is not None
])
y_temp_train = train_temp["label"].values

X_temp_test = np.vstack([
    fp_to_array(mol_from_smiles(s))
    for s in test_temp["canonical_smiles"]
    if mol_from_smiles(s) is not None
])
y_temp_test = test_temp["label"].values

rf_temporal = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf_temporal.fit(X_temp_train, y_temp_train)
pred_temporal = rf_temporal.predict_proba(X_temp_test)[:, 1]

temporal_auc = roc_auc_score(y_temp_test, pred_temporal) if len(np.unique(y_temp_test)) > 1 else np.nan
temporal_acc = accuracy_score(y_temp_test, (pred_temporal >= 0.5).astype(int))

results = pd.DataFrame([
    {
        "split": "Random",
        "roc_auc": round(random_auc, 4),
        "accuracy": round(random_acc, 4),
        "n_test": len(y_test)
    },
    {
        "split": "Scaffold",
        "roc_auc": round(scaffold_auc, 4) if pd.notna(scaffold_auc) else np.nan,
        "accuracy": round(scaffold_acc, 4),
        "n_test": len(y_sc_test)
    },
    {
        "split": "Temporal",
        "roc_auc": round(temporal_auc, 4) if pd.notna(temporal_auc) else np.nan,
        "accuracy": round(temporal_acc, 4),
        "n_test": len(y_temp_test)
    }
])

results.to_csv(OUTPUT_FILE, index=False)

In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

INPUT_FILE = "egfr_ic50_full.csv"
SMILES_FILE = "module10_molecule_smiles_lookup.csv"

df = pd.read_csv(INPUT_FILE)
df.columns = df.columns.str.lower().str.strip()

df = df[df["standard_type"].astype(str).str.upper() == "IC50"].copy()
df["standard_value"] = pd.to_numeric(df["standard_value"], errors="coerce")
df = df.dropna(subset=["standard_value"]).copy()
df["ic50_nm"] = df["standard_value"]

mol_activity = df.groupby("molecule_chembl_id", as_index=False).agg(
    median_ic50_nm=("ic50_nm", "median")
)
mol_activity["label"] = (mol_activity["median_ic50_nm"] <= 1000).astype(int)

smiles_df = pd.read_csv(SMILES_FILE)
smiles_df.columns = smiles_df.columns.str.lower().str.strip()

data = mol_activity.merge(smiles_df, on="molecule_chembl_id", how="left")
data = data.dropna(subset=["canonical_smiles"]).copy()

morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def mol_from_smiles(s):
    try:
        return Chem.MolFromSmiles(s)
    except:
        return None

def fp_to_array(mol):
    fp = morgan_gen.GetFingerprint(mol)
    arr = np.zeros((2048,), dtype=int)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

valid_rows = []
fp_rows = []

for _, row in data.iterrows():
    mol = mol_from_smiles(row["canonical_smiles"])
    if mol is not None:
        valid_rows.append(row)
        fp_rows.append(fp_to_array(mol))

data = pd.DataFrame(valid_rows).reset_index(drop=True)
X = np.vstack(fp_rows)
y = data["label"].values

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, data.index.values, test_size=0.2, random_state=42, stratify=y
)

ensemble_models = []
for seed in range(5):
    rf = RandomForestClassifier(
        n_estimators=300,
        random_state=42 + seed,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)
    ensemble_models.append(rf)

ensemble_probs = np.column_stack([m.predict_proba(X_test)[:, 1] for m in ensemble_models])
ensemble_mean = ensemble_probs.mean(axis=1)
ensemble_std = ensemble_probs.std(axis=1)
ensemble_conf = np.maximum(ensemble_mean, 1 - ensemble_mean)
ensemble_pred = np.where(ensemble_mean >= 0.5, "Active", "Inactive")

ensemble_out = data.loc[idx_test, ["molecule_chembl_id", "canonical_smiles"]].reset_index(drop=True)
ensemble_out["prediction"] = ensemble_pred
ensemble_out["confidence"] = np.round(ensemble_conf * 100, 2)
ensemble_out["mean_probability_active"] = np.round(ensemble_mean, 4)
ensemble_out["uncertainty_std"] = np.round(ensemble_std, 4)
ensemble_out.to_csv("module12_ensemble_uncertainty.csv", index=False)

base_model = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
base_model.fit(X_train, y_train)

mc_probs = []
for _ in range(30):
    sample_idx = np.random.choice(len(X_test), size=len(X_test), replace=True)
    X_sample = X_test[sample_idx]
    mc_probs.append(base_model.predict_proba(X_sample)[:, 1])

mc_probs = np.array(mc_probs)
mc_mean = mc_probs.mean(axis=0)
mc_std = mc_probs.std(axis=0)
mc_conf = np.maximum(mc_mean, 1 - mc_mean)
mc_pred = np.where(mc_mean >= 0.5, "Active", "Inactive")

mc_out = data.loc[idx_test, ["molecule_chembl_id", "canonical_smiles"]].reset_index(drop=True)
mc_out["prediction"] = mc_pred
mc_out["confidence"] = np.round(mc_conf * 100, 2)
mc_out["mean_probability_active"] = np.round(mc_mean, 4)
mc_out["uncertainty_std"] = np.round(mc_std, 4)
mc_out.to_csv("module12_mc_dropout_uncertainty.csv", index=False)

if (
    len(ensemble_out) > 0 and
    len(mc_out) > 0 and
    "module12_ensemble_uncertainty.csv" and
    "module12_mc_dropout_uncertainty.csv"
):
    print("Task completed")
else:
    print("Task not completed")

In [ ]:
!git clone https://github.com/oriondollar/TransVAE.git
%cd TransVAE

!pip install -q selfies
!pip install -q torch torchvision torchaudio

In [ ]:
!ls
!find . -maxdepth 2 -type f | sed 's|^\./||' | grep -E 'sample|ckpt|checkpoint|pretrain|vocab|weights'

In [ ]:
from pathlib import Path

p = Path("/content/TransVAE/transvae/trans_models.py")
txt = p.read_text()
old = "loaded_checkpoint = torch.load(checkpoint_path, map_location=torch.device('cpu'))"
new = "loaded_checkpoint = torch.load(checkpoint_path, map_location=torch.device('cpu'), weights_only=False)"
if old in txt:
    p.write_text(txt.replace(old, new))
    print("Patched torch.load to weights_only=False")
else:
    print("Line not found")

In [ ]:
import os
import pandas as pd

INPUT_FILE = "/content/module10_molecule_smiles_lookup.csv"
SEED_FILE = "/content/TransVAE/egfr_seed_smiles.txt"
FINAL_OUTPUT = "/content/module8_transvae_generated_molecules.csv"

df = pd.read_csv(INPUT_FILE)

seed_col = "canonical_smiles" if "canonical_smiles" in df.columns else "smiles"
seed_smiles = df[seed_col].dropna().astype(str).unique().tolist()

with open(SEED_FILE, "w") as f:
    for smi in seed_smiles:
        f.write(smi.strip() + "\n")

%cd /content/TransVAE

!PYTHONPATH=/content/TransVAE python scripts/sample.py \
    --model transvae \
    --model_ckpt checkpoints/trans4x-256_zinc.ckpt \
    --mols egfr_seed_smiles.txt \
    --sample_mode rand \
    --n_samples 200 \
    --save_path generated_samples.txt

In [ ]:
import os
import pandas as pd

seed_df = pd.read_csv("/content/module10_molecule_smiles_lookup.csv")
seed_col = "canonical_smiles" if "canonical_smiles" in seed_df.columns else "smiles"
seed_smiles = set(seed_df[seed_col].dropna().astype(str))

out_file = "/content/TransVAE/generated_samples.txt"
gen = []

if os.path.exists(out_file):
    with open(out_file, "r") as f:
        for line in f:
            smi = line.strip()
            if smi and smi not in seed_smiles:
                gen.append(smi)

gen = list(dict.fromkeys(gen))[:100]

final_df = pd.DataFrame({"generated_smiles": gen})
final_df.to_csv("/content/module8_transvae_generated_molecules.csv", index=False)

print("Saved:", "/content/module8_transvae_generated_molecules.csv")
print("Generated molecules:", len(final_df))
print(final_df.head(10))

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
!pip install rdkit pandas numpy -q

import pandas as pd
import random
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, Lipinski, QED

# Load input CSV
df = pd.read_csv("/content/module10_molecule_smiles_lookup.csv")

# Keep only valid SMILES
def is_valid(smiles):
    return Chem.MolFromSmiles(smiles) is not None

df = df[df["canonical_smiles"].apply(is_valid)].copy()
seed_smiles = df["canonical_smiles"].tolist()

# Simple scoring function
def score_molecule(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return 0

    mw = Descriptors.MolWt(mol)
    logp = Crippen.MolLogP(mol)
    hbd = Lipinski.NumHDonors(mol)
    hba = Lipinski.NumHAcceptors(mol)
    qed = QED.qed(mol)

    activity = 1 if 200 <= mw <= 600 else 0
    admet = 1 if logp < 5 and hbd <= 5 and hba <= 10 else 0
    synthesis = qed

    return 0.4 * activity + 0.3 * admet + 0.3 * synthesis

# Very simple mutation: add one atom
def mutate_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    rw = Chem.RWMol(mol)
    try:
        atom_idx = random.randint(0, rw.GetNumAtoms() - 1)
        new_atom = Chem.Atom(random.choice([6, 7, 8]))  # C, N, O
        new_idx = rw.AddAtom(new_atom)
        rw.AddBond(atom_idx, new_idx, Chem.BondType.SINGLE)
        new_mol = rw.GetMol()
        Chem.SanitizeMol(new_mol)
        return Chem.MolToSmiles(new_mol)
    except:
        return None

# RL-style simple loop
results = []

for smi in seed_smiles[:200]:   # use first 200 seeds for speed
    new_smi = mutate_smiles(smi)
    if new_smi:
        old_score = score_molecule(smi)
        new_score = score_molecule(new_smi)
        if new_score >= old_score:
            results.append([smi, new_smi, old_score, new_score])

# Save results
out_df = pd.DataFrame(results, columns=["original_smiles", "optimized_smiles", "old_score", "new_score"])
out_df = out_df.sort_values("new_score", ascending=False)
out_df.to_csv("module9_optimized_output.csv", index=False)

print(out_df.head(20))
print("Saved: module9_optimized_output.csv")